# 📝 텍스트 임베딩 과제 LV1 정답 — 임베딩·유사도·검색·군집 기초 (강사용)

각 문제의 **모범답안 + 해설(접근법·흔한 실수)** 입니다. 학생이 스스로 풀어 본 뒤 비교하도록 안내하세요.

- 경로는 정답 노트북 기준 `../../day12_텍스트임베딩/data/` 입니다.
- 임베딩은 노트북 안에서 `model.encode(...)` 로 직접 만듭니다(헤드라인 20건이라 빠릅니다). `encode` 는 결정적이라 매번 같은 값이 나옵니다.
- 코사인 값은 실측값을 감싸는 **범위**로 채점하고, 실루엣·군집처럼 흔들릴 수 있는 값도 **범위·방향**으로 채점합니다.

아래 두 셀을 먼저 실행해 라이브러리와 임베딩 모델을 준비하세요.

In [ ]:
# [제공 코드] 이 단원에 필요한 라이브러리를 준비합니다.
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from umap import UMAP
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

이어서 이 단원에서 배운 **한국어 임베딩 모델**을 불러옵니다(문장→768차원 벡터). 처음 부를 때 모델을 내려받느라 조금 걸릴 수 있어요.

In [ ]:
# [제공 코드] 이 단원에서 배운 한국어 임베딩 모델을 불러옵니다(문장->768차원 벡터).
model = SentenceTransformer('jhgan/ko-sroberta-multitask')

## 데이터 살펴보기 — 먼저 데이터를 이해합니다
문제를 풀기 전에 **어떤 데이터인지 먼저 파악**합니다. `head()` 로 앞부분을, `info()` 로 열·자료형·결측을, `value_counts()` 로 분야 분포를 봅니다. (아래 셀은 실행만 하면 됩니다.)

In [ ]:
# [제공 코드] 데이터를 먼저 살펴봅니다 — 앞부분·구조·분야 분포
df = pd.read_csv('../../day12_텍스트임베딩/data/news_headlines.csv')
print("행·열 크기:", df.shape)
print("\n[앞 5행] head()"); display(df.head())
print("\n[열·자료형·결측] info()"); df.info()
print("\n[분야(category) 분포] value_counts()"); display(df["category"].value_counts().to_frame("건수"))

## 1. 헤드라인 한 개를 벡터로 — 임베딩 만들기
**배경**: 임베딩의 출발점은 **문장 하나를 벡터 하나로** 바꾸는 것입니다. 모델의 `encode` 에 문장을 넣으면 그 문장의 뜻을 담은 768차원 벡터가 나옵니다.

**요구사항**:
- `df` 의 **0번 헤드라인**(`df.loc[0, 'headline']`)을 `model.encode(...)` 로 임베딩해 변수 `vec` 에 담으세요.
- `vec.shape` 가 `(768,)` 인지 확인하세요(문장 한 개는 768차원 벡터 하나).

**예시**
```
vec.shape  →  (768,)
```
<details><summary>힌트</summary>

```text
접근방법:
- 헤드라인 문자열 하나를 임베딩 모델의 encode 에 넣어 벡터를 얻는다.

세부구현:
1. df 의 0번 행 headline 값을 꺼낸다
2. model 의 encode 에 그 문장을 넣어 vec 에 담는다
3. vec 의 shape 를 출력해 (768,) 인지 확인한다
```

</details>

In [ ]:
vec = model.encode(df.loc[0, 'headline'])
print("벡터 모양:", vec.shape)

In [ ]:
# [자가채점]
assert vec.shape == (768,)
print("✅ 문제1 통과!")

### 해설 — 문제 1
- **접근법**: `encode` 에 문자열 하나를 넣으면 `(768,)` 모양의 1차원 벡터가 나옵니다. 이 768개의 숫자가 문장의 '의미'를 좌표로 담고 있어, 뜻이 비슷한 문장은 벡터도 가까워집니다.
- **흔한 실수**: 문자열을 리스트로 감싸(`model.encode([문장])`) 넣으면 `(1, 768)` 모양의 2차원이 나옵니다. 이 문제는 문장 하나만 넣어 `(768,)` 을 만듭니다.

## 2. 헤드라인 20건을 한꺼번에 임베딩
**배경**: 분석하려면 헤드라인 **전부**를 벡터로 바꿔 둬야 합니다. `encode` 에 문장 **리스트**를 넣으면 `(문장 수, 768)` 모양의 행렬이 한 번에 나옵니다.

**요구사항**:
- `df['headline']` 의 헤드라인 20건을 리스트로 만들어 `model.encode(...)` 로 임베딩하고, 결과를 변수 `embeddings` 에 담으세요.
- `embeddings.shape` 가 `(20, 768)` 인지 확인하세요(헤드라인 20건 × 각 768차원).
- 이 `embeddings` 는 **뒤 문제(3~15번)에서 그대로** 사용합니다.

**예시**
```
embeddings.shape  →  (20, 768)
```
<details><summary>힌트</summary>

```text
접근방법:
- 헤드라인 열을 리스트로 바꿔 통째로 encode 에 넣으면 문장마다 한 행인 행렬이 나온다.

세부구현:
1. df 의 headline 열을 리스트로 만든다
2. model 의 encode 에 그 리스트를 넣어 embeddings 에 담는다
3. embeddings 의 shape 를 출력해 (20, 768) 인지 확인한다
```

</details>

In [ ]:
embeddings = model.encode(df['headline'].tolist())
print("임베딩 행렬 모양:", embeddings.shape)

In [ ]:
# [자가채점]
assert embeddings.shape == (20, 768)
print("✅ 문제2 통과!")

### 해설 — 문제 2
- **접근법**: 문장 리스트를 넣으면 `encode` 가 각 문장을 한 행으로 쌓아 `(20, 768)` 행렬을 줍니다. 행 하나가 헤드라인 하나의 벡터입니다(`embeddings[i]`).
- **흔한 실수**: `.tolist()` 를 빼고 `df['headline']`(시리즈)을 그대로 넣는 것입니다. `encode` 는 **문자열 또는 문자열 리스트**만 받으므로 시리즈를 넣으면 `ValueError: Unsupported input type: Series` 가 납니다 — `.tolist()` 로 리스트를 만들어 넘기세요.

## 3. 코사인 유사도 — 같은 분야 두 헤드라인
**배경**: 두 문장이 얼마나 비슷한지는 **코사인 유사도**(두 벡터가 이루는 각도, 1에 가까울수록 비슷)로 잽니다. 먼저 **같은 분야**(둘 다 스포츠)인 10번·11번 헤드라인의 유사도를 재 봅니다.

**요구사항**:
- 문제 2의 `embeddings` 에서 10번·11번 벡터로 코사인 유사도를 구해 변수 `sim_same` 에 담으세요. `cosine_similarity(embeddings[10:11], embeddings[11:12])[0][0]` 로 값 하나를 꺼냅니다.
- `sim_same` 은 약 **0.31** 입니다. 모델 버전에 따라 소수 자리가 미세하게 달라질 수 있어 **`0.28 ~ 0.34` 범위**로 채점합니다(정확한 값이 아니라 범위·방향으로 봅니다).

**예시**
```
round(sim_same, 2)  →  0.31   (스포츠 ↔ 스포츠)
```
<details><summary>힌트</summary>

```text
접근방법:
- 두 헤드라인의 벡터를 코사인 유사도 함수에 넣어 값 하나를 꺼낸다.

세부구현:
1. embeddings 에서 10번·11번 벡터를 각각 한 행짜리로 고른다
2. cosine_similarity 에 두 벡터를 넣고 [0][0] 으로 값 하나를 꺼내 sim_same 에 담는다
3. 값을 출력해 약 0.31 인지 확인한다
```

</details>

In [ ]:
sim_same = cosine_similarity(embeddings[10:11], embeddings[11:12])[0][0]
print("같은 분야(스포츠↔스포츠) 유사도:", round(sim_same, 2))

In [ ]:
# [자가채점]
assert 0.28 < float(sim_same) < 0.34
print("✅ 문제3 통과!")

### 해설 — 문제 3
- **접근법**: `cosine_similarity(A, B)` 는 A·B 의 각 행 사이 유사도 **행렬**을 줍니다. 한 쌍만 필요하므로 `[0][0]` 으로 값 하나를 꺼냅니다. 10·11번은 둘 다 스포츠라 유사도가 비교적 높습니다.
- **흔한 실수**: `embeddings[10]` 은 1차원 `(768,)` 이라 그대로 넣으면 에러가 납니다. `embeddings[10:11]` 처럼 **한 행짜리 2차원**(`(1, 768)`)으로 잘라 넣으세요.

## 4. 코사인 유사도 — 다른 분야 두 헤드라인
**배경**: 이번엔 **다른 분야**(스포츠 vs IT)인 10번·15번 헤드라인의 유사도를 재고, 문제 3의 **같은 분야** 유사도와 비교합니다. 임베딩이 분야(뜻)의 차이를 담았다면 같은 분야가 더 높아야 합니다.

**요구사항**:
- 10번·15번 벡터로 코사인 유사도를 구해 변수 `sim_diff` 에 담으세요.
- `sim_diff` 는 약 **0.13** 입니다(**`0.08 ~ 0.18` 범위**로 채점).
- 문제 3의 `sim_same`(같은 분야)이 `sim_diff`(다른 분야)보다 **큰지** 확인하세요(같은 분야가 더 비슷) — 이 방향이 이 문제의 핵심입니다.

**예시**
```
round(sim_diff, 2)      →  0.13    (스포츠 ↔ IT)
sim_same > sim_diff     →  True    (같은 분야가 더 비슷)
```
<details><summary>힌트</summary>

```text
접근방법:
- 문제 3과 똑같은 방법으로 대상만 바꿔(10·15번) 유사도를 구하고, 같은 분야 값과 크기를 비교한다.

세부구현:
1. embeddings 에서 10번·15번 벡터를 한 행짜리로 고른다
2. cosine_similarity 로 값 하나를 꺼내 sim_diff 에 담는다
3. sim_same 과 sim_diff 를 출력해 같은 분야가 더 큰지 확인한다
```

</details>

In [ ]:
sim_diff = cosine_similarity(embeddings[10:11], embeddings[15:16])[0][0]
print("다른 분야(스포츠↔IT) 유사도:", round(sim_diff, 2))
print("같은 분야가 더 비슷한가?", bool(sim_same > sim_diff))

In [ ]:
# [자가채점]
assert 0.08 < float(sim_diff) < 0.18
assert sim_same > sim_diff
print("✅ 문제4 통과!")

### 해설 — 문제 4
- **접근법**: 같은 절차로 대상만 바꿉니다. 스포츠↔IT 는 뜻이 멀어 유사도(0.13)가 스포츠↔스포츠(0.313)보다 낮습니다. 이렇게 **같은 분야 > 다른 분야**가 나오면 임베딩이 분야 차이를 잘 담은 것입니다.
- **흔한 실수**: 유사도 값이 작다고 '틀렸다'고 오해하기 쉽습니다. 코사인 유사도는 **상대 비교**가 핵심이라, 같은 분야와 다른 분야의 **차이**를 보는 것이 중요합니다.

## 5. 의미 검색 — 질문에 가장 가까운 헤드라인 1건
**배경**: **의미 검색**은 질문을 임베딩해, 모든 문서 중 **뜻이 가장 가까운** 것을 찾는 것입니다. 질문의 단어가 헤드라인에 그대로 없어도 뜻이 맞으면 찾아냅니다.

**요구사항**:
- 질문 문자열 `query = '프로 스포츠 경기에서 우승'` 을 `model.encode([query])` 로 임베딩하세요.
- 질문 벡터와 `embeddings` 의 코사인 유사도 배열(길이 20)을 구해 변수 `scores` 에 담으세요(`cosine_similarity(query_vec, embeddings)[0]`).
- 유사도가 **가장 큰** 헤드라인의 인덱스를 정수로 변수 `top1` 에 담으세요. `top1` 은 **11**(스포츠 헤드라인)이 나와야 합니다.

**예시**
```
len(scores)  →  20
top1         →  11   (가장 가까운 헤드라인의 인덱스)
```
<details><summary>힌트</summary>

```text
접근방법:
- 질문을 문서와 같은 모델로 임베딩하고, 모든 문서와의 유사도를 구해 가장 큰 값의 위치를 찾는다.

세부구현:
1. query 를 리스트로 감싸 encode 해 질문 벡터를 만든다
2. cosine_similarity 로 질문 벡터와 embeddings 의 유사도 배열을 구해 scores 에 담는다
3. 유사도가 가장 큰 위치를 정수로 top1 에 담는다(가장 큰 값의 인덱스를 찾는 numpy 함수 사용)
```

</details>

In [ ]:
query = '프로 스포츠 경기에서 우승'
query_vec = model.encode([query])
scores = cosine_similarity(query_vec, embeddings)[0]
top1 = int(np.argmax(scores))
print("질문:", query)
print("가장 가까운 헤드라인:", top1, "|", df.loc[top1, "headline"], "| 유사도", round(scores[top1], 3))

In [ ]:
# [자가채점]
assert len(scores) == 20
assert top1 == 11
print("✅ 문제5 통과!")

### 해설 — 문제 5
- **접근법**: 질문도 문서와 **같은 모델**로 임베딩해야 같은 의미 공간에 놓입니다. `np.argmax(scores)` 는 유사도가 가장 큰 위치를 줍니다. 질문의 '프로·우승' 이 11번 헤드라인('프로야구 한국시리즈, 안방에서 우승 확정 짓나')과 뜻이 가장 가까워 뽑힙니다.
- **흔한 실수**: `query` 를 리스트로 감싸지 않고 `model.encode(query)` 로 넣으면 벡터가 1차원이라 `cosine_similarity` 에 바로 못 넣습니다. `[query]` 처럼 리스트로 감싸세요.

## 6. 의미 검색 — 질문에 가장 가까운 헤드라인 top3
**배경**: 이번엔 **다른 질문**으로 가장 가까운 헤드라인 **3건**을 순서대로 찾습니다. 상위 k개를 뽑는 것이 실제 검색과 더 가깝습니다.

**요구사항**:
- 질문 `query2 = '인공지능 기술과 반도체 신제품'` 을 임베딩하고, `embeddings` 와의 유사도 배열을 변수 `scores2` 에 담으세요.
- 유사도가 **큰 순서**로 정렬한 인덱스에서 **상위 3개**를 변수 `top3` 에 담으세요(`np.argsort(-scores2)[:3]`).
- `top3` 의 길이는 3이고, **첫 번째**(가장 가까운) 인덱스는 **15**(IT 헤드라인)가 나와야 합니다.

**예시**
```
len(top3)   →  3
top3[0]     →  15   (가장 가까운 헤드라인의 인덱스)
```
<details><summary>힌트</summary>

```text
접근방법:
- 질문을 임베딩해 유사도 배열을 구하고, 큰 값부터 정렬한 인덱스에서 앞 3개를 고른다.

세부구현:
1. query2 를 리스트로 감싸 encode 하고 유사도 배열 scores2 를 구한다
2. 유사도에 마이너스를 붙여 argsort 하면 큰 값이 앞으로 온다(내림차순 인덱스)
3. 그 인덱스의 앞 3개를 top3 에 담는다
```

</details>

In [ ]:
query2 = '인공지능 기술과 반도체 신제품'
query_vec2 = model.encode([query2])
scores2 = cosine_similarity(query_vec2, embeddings)[0]
top3 = np.argsort(-scores2)[:3]
print("질문:", query2)
for rank, i in enumerate(top3, start=1):
    print(f"  {rank}위  #{i}  유사도 {scores2[i]:.3f}  [{df.loc[i, 'category']}] {df.loc[i, 'headline']}")

In [ ]:
# [자가채점]
assert len(top3) == 3
assert int(top3[0]) == 15
print("✅ 문제6 통과!")

### 해설 — 문제 6
- **접근법**: 문제 5와 같은 검색이지만 상위 1개가 아니라 **top3** 를 뽑습니다. `np.argsort` 는 기본이 오름차순이라, 유사도에 **마이너스**를 붙여 정렬하면 큰 값이 앞으로 옵니다. 앞 3개가 top-3 입니다.
- **흔한 실수**: `np.argsort(scores2)[:3]` (마이너스 없이)은 **가장 안 비슷한** 3개를 줍니다. 내림차순을 위해 부호를 뒤집으세요.

## 7. 차원축소 — UMAP으로 768차원을 2차원으로
**배경**: 768차원 벡터는 눈으로 볼 수 없습니다. **UMAP** 은 각 점의 **가까운 이웃 관계가 유지되도록** 임베딩을 **2차원 좌표**로 줄여 줍니다(나중에 산점도로 그릴 수 있게).

**요구사항**:
- `UMAP(n_components=2, n_neighbors=5, min_dist=0.05, random_state=0)` 로 `embeddings` 를 2차원으로 변환해 결과를 변수 `coords` 에 담으세요(`random_state=0` 을 주면 매번 같은 좌표가 나옵니다).
- `coords.shape` 가 `(20, 2)` 인지 확인하세요(헤드라인 20건 × 각 2차원 좌표).

**예시**
```
coords.shape  →  (20, 2)
```
<details><summary>힌트</summary>

```text
접근방법:
- UMAP 을 2차원으로 설정하고 임베딩에 fit_transform 을 적용해 2D 좌표를 얻는다.

세부구현:
1. UMAP 을 n_components=2, n_neighbors=5, min_dist=0.05, random_state=0 으로 만든다
2. fit_transform 에 embeddings 를 넣어 coords 에 담는다
3. coords 의 shape 를 출력해 (20, 2) 인지 확인한다
```

</details>

In [ ]:
coords = UMAP(n_components=2, n_neighbors=5, min_dist=0.05,
              random_state=0).fit_transform(embeddings)
print("UMAP 좌표 모양:", coords.shape)

In [ ]:
# [자가채점]
assert coords.shape == (20, 2)
print("✅ 문제7 통과!")

### 해설 — 문제 7
- **접근법**: `UMAP(n_components=2, ...)` 는 '2차원으로 줄이겠다'는 뜻입니다. `fit_transform` 은 각 헤드라인을 2개의 좌표로 바꿔 `(20, 2)` 행렬을 줍니다. 이 좌표로 산점도를 그리면 비슷한 헤드라인이 가까이 모입니다.
- **흔한 실수**: `fit`(변환 없이 학습만)과 `fit_transform`(학습+변환)을 헷갈리기 쉽습니다. 좌표가 필요하므로 `fit_transform` 을 쓰세요.

## 8. 한 헤드라인의 2차원 좌표 살펴보기
**배경**: UMAP 으로 줄인 `coords` 에서 **한 헤드라인의 좌표**는 한 행입니다. 0번 헤드라인의 2D 좌표를 꺼내, 좌표가 정말 **2개의 숫자**(x, y)로 되어 있는지 확인해 봅니다.

**요구사항**:
- 문제 7의 `coords` 에서 **0번 헤드라인의 좌표**를 꺼내 변수 `first_coord` 에 담으세요(`coords[0]`).
- `len(first_coord)` 가 **2** 인지 확인하세요(2차원이므로 좌표 값이 2개).

**예시**
```
len(first_coord)  →  2      (x, y 두 개의 좌표 값)
```
<details><summary>힌트</summary>

```text
접근방법:
- 2D 좌표 행렬에서 0번 행을 꺼내면 그 헤드라인의 (x, y) 좌표가 된다. 원소 개수를 센다.

세부구현:
1. coords 의 0번 행을 first_coord 에 담는다
2. len 으로 좌표 값의 개수를 확인한다(2여야 함)
```

</details>

In [ ]:
first_coord = coords[0]
print("0번 헤드라인의 2D 좌표:", np.round(first_coord, 3), "  좌표 값 개수:", len(first_coord))

In [ ]:
# [자가채점]
assert len(first_coord) == 2
print("✅ 문제8 통과!")

### 해설 — 문제 8
- **접근법**: `coords[0]` 은 0번 헤드라인의 2차원 좌표 `(x, y)` 입니다. `len` 이 2인 것은 '2차원으로 줄였다'는 사실을 그대로 보여 줍니다. 이 x·y 가 산점도의 가로·세로 위치가 됩니다.
- **흔한 실수**: `coords[0]` (한 행, 좌표 하나)과 `coords[:, 0]` (모든 행의 x 좌표, 20개)은 다릅니다. 이 문제는 **한 헤드라인의 좌표** `coords[0]` 을 봅니다.

## 9. 군집화 — KMeans로 헤드라인 그룹 만들기
**배경**: 정답(분야) 없이 비슷한 헤드라인끼리 묶는 것이 **군집화**입니다. **KMeans** 는 미리 정한 개수(k)의 중심을 찾아 각 헤드라인을 가장 가까운 중심에 배정합니다. 768차원 원본은 노이즈가 커서 군집이 흐릿하므로, 문제 7에서 만든 **2차원 좌표 `coords`** 위에서 묶습니다. 분야는 4개지만 **몇 개로 나누는 게 좋은지는 실루엣으로 정합니다**(문제 12) — 여기서는 우선 **k=3** 으로 나눠 봅니다.

**요구사항**:
- 문제 7의 `coords` 를 사용합니다. `KMeans(n_clusters=3, random_state=0, n_init=10)` 로 모델을 만들고 `fit_predict(coords)` 로 각 헤드라인의 군집 번호를 구해 변수 `labels` 에 담으세요.
- `labels` 의 길이는 **20**(헤드라인 수), 서로 다른 군집 번호의 개수(`len(set(labels))`)는 **3** 이어야 합니다.

**예시**
```
len(labels)        →  20
len(set(labels))   →  3    (군집이 3개)
```
<details><summary>힌트</summary>

```text
접근방법:
- 2차원 좌표 coords 에 KMeans 를 학습시켜 각 헤드라인의 군집 번호를 얻는다. 재현을 위해 random_state 를 고정한다.

세부구현:
1. KMeans 를 n_clusters=3, random_state=0, n_init=10 으로 만든다
2. fit_predict 에 coords 를 넣어 labels 에 담는다
3. labels 의 길이와 서로 다른 군집 번호의 개수를 확인한다
```

</details>

In [ ]:
labels = KMeans(n_clusters=3, random_state=0, n_init=10).fit_predict(coords)
print("군집 라벨:", labels.tolist())
print("헤드라인 수:", len(labels), " 군집 개수:", len(set(labels)))

In [ ]:
# [자가채점]
assert len(labels) == 20
assert len(set(labels)) == 3
print("✅ 문제9 통과!")

### 해설 — 문제 9
- **접근법**: `fit_predict` 는 군집을 학습하고 각 점의 군집 번호를 한 번에 돌려줍니다. `n_init=10` 은 서로 다른 초기 중심으로 10번 시도해 가장 좋은 결과를 고른다는 뜻이고, `random_state=0` 으로 결과를 고정합니다. 768차원 원본 대신 **UMAP으로 줄인 `coords`** 에서 묶어야 군집이 또렷합니다.
- **흔한 실수**: `random_state` 를 안 주면 실행마다 군집 번호가 달라질 수 있습니다. 채점을 위해 반드시 고정하세요. 또 군집 **번호 자체**(0/1/2)는 실행 방식에 따라 뒤바뀔 수 있어, 여기서는 **개수**만 봅니다.

## 10. 같은 분야 헤드라인이 같은 군집에 묶였나
**배경**: 군집이 분야를 잘 잡았다면 **같은 분야** 헤드라인은 **같은 군집**에 묶여야 합니다. 문제 9의 `labels` 로, 스포츠 헤드라인인 10번과 11번이 같은 군집인지 확인해 봅니다.

**요구사항**:
- 문제 9의 `labels` 를 그대로 사용하세요.
- 10번 헤드라인의 군집 번호(`labels[10]`)와 11번 헤드라인의 군집 번호(`labels[11]`)가 **같은지** 비교한 결과(참/거짓)를 변수 `same_cluster` 에 담으세요.
- 두 헤드라인은 모두 스포츠라 같은 군집에 묶여, `same_cluster` 는 **참(True)** 이어야 합니다.

**예시**
```
same_cluster  →  True   (10번·11번이 같은 군집)
```
<details><summary>힌트</summary>

```text
접근방법:
- 두 헤드라인의 군집 번호를 꺼내 서로 같은지 비교한다.

세부구현:
1. labels 에서 10번·11번의 군집 번호를 각각 꺼낸다
2. 두 값이 같은지 비교(==)한 결과를 same_cluster 에 담는다
```

</details>

In [ ]:
same_cluster = labels[10] == labels[11]
print("10번 군집:", labels[10], " 11번 군집:", labels[11], " 같은 군집?", bool(same_cluster))

In [ ]:
# [자가채점]
assert bool(same_cluster) == True
assert same_cluster == (labels[10] == labels[11])   # 값을 지어내지 않고 labels 로 구했는지
print("✅ 문제10 통과!")

### 해설 — 문제 10
- **접근법**: 군집 번호 자체는 실행에 따라 뒤바뀔 수 있지만, '같은 분야 두 헤드라인이 **한 군집에** 있는가'는 안정적으로 확인할 수 있습니다. 10·11번은 둘 다 스포츠라 임베딩이 가까워 같은 군집에 묶입니다.
- **흔한 실수**: `labels[10]` 의 **값**(예: 2)을 정답이라고 외우면 안 됩니다. 그 번호는 실행마다 달라질 수 있어, 두 헤드라인의 번호가 **서로 같은지**를 비교하는 것이 핵심입니다.

## 11. 군집이 잘 나뉘었나 — 실루엣 점수
**배경**: 군집이 **얼마나 잘 나뉘었는지**는 **실루엣 점수**로 잽니다(−1~1, 클수록 좋음). 같은 군집끼리 가깝고 다른 군집과 멀수록 높습니다. 문제 9의 k=3 군집이 얼마나 잘 나뉘었는지 확인해 봅니다.

**요구사항**:
- 문제 7의 `coords` 와 문제 9의 `labels` 로 실루엣 점수를 구해 변수 `sil` 에 담으세요(`silhouette_score(coords, labels)`).
- `sil` 은 대략 **0.5 안팎**의 양수 값이 나옵니다(정확한 값이 아니라 범위로 채점합니다).

**예시**
```
0.3 < sil < 0.8   →  True   (양수이고 0.5 안팎)
```
<details><summary>힌트</summary>

```text
접근방법:
- 2차원 좌표와 군집 라벨을 실루엣 점수 함수에 넣는다. 값이 클수록 군집이 뚜렷하게 나뉜 것.

세부구현:
1. silhouette_score 에 coords 와 labels 를 넣어 sil 에 담는다
2. sil 을 출력해 양수인지 확인한다
```

</details>

In [ ]:
sil = silhouette_score(coords, labels)
print("실루엣 점수:", round(sil, 3))

In [ ]:
# [자가채점]
assert 0.3 < sil < 0.8
print("✅ 문제11 통과!")

### 해설 — 문제 11
- **접근법**: 실루엣 점수는 각 점이 '자기 군집에 얼마나 잘 맞는지(응집)'와 '가장 가까운 다른 군집과 얼마나 떨어졌는지(분리)'를 함께 봅니다. UMAP으로 줄인 `coords`(2차원)에서 재면 약 0.5 로 나옵니다. 다만 이 크기는 **UMAP 이 덩어리를 또렷하게 펼쳐 놓은 좌표에서 잰 값**이라 부풀려져 있습니다 — 768차원 원본에서 재면 0.04 수준입니다. 그래서 **크기보다 여러 k 사이의 순위**(다음 문제)를 봅니다.
- **흔한 실수**: 실루엣 점수는 군집이 2개 이상일 때만 정의됩니다(전부 한 군집이면 에러). 또 값이 작다고 군집이 '틀린' 것은 아닙니다 — 다음 문제처럼 **여러 k 를 비교**해 상대적으로 나은 쪽을 고릅니다.

## 12. 몇 개로 나눌까 — k=3 과 k=6 실루엣 비교
**배경**: 군집을 **몇 개(k)로 나눌지**는 실루엣 점수로 비교해 정합니다. 문제 7의 `coords` 위에서 k=3 과 k=6 으로 각각 군집화해 실루엣 점수를 재고, **너무 잘게 쪼개면 오히려 나빠지는지** 확인해 봅니다.

**요구사항**:
- 문제 7의 `coords` 를 사용합니다. `KMeans(n_clusters=3, random_state=0, n_init=10)` 로 군집화한 라벨의 실루엣 점수를 변수 `sil3` 에 담으세요(문제 11의 값과 같습니다).
- `KMeans(n_clusters=6, random_state=0, n_init=10)` 로 군집화한 라벨의 실루엣 점수를 변수 `sil6` 에 담으세요.
- 두 값을 비교해, k=3 쪽이 더 큰지(`sil3 > sil6`)를 변수 `k3_better` 에 담으세요. 이 데이터에서는 **k=3 이 더 큽니다**(`k3_better` 는 참) — 헤드라인이 20건뿐인데 6개로 쪼개면 군집이 잘게 부서집니다.

**예시**
```
k3_better  →  True   (k=3 의 실루엣이 k=6 보다 큼)
```
<details><summary>힌트</summary>

```text
접근방법:
- 같은 coords 에 k=3, k=6 으로 각각 군집화해 실루엣 점수를 구하고, 두 점수의 크기를 비교한다.

세부구현:
1. k=3 으로 fit_predict 한 라벨로 silhouette_score 를 구해 sil3 에 담는다
2. k=6 으로 fit_predict 한 라벨로 silhouette_score 를 구해 sil6 에 담는다
3. sil3 가 sil6 보다 큰지 비교한 결과를 k3_better 에 담는다
```

</details>

In [ ]:
labels3 = KMeans(n_clusters=3, random_state=0, n_init=10).fit_predict(coords)
sil3 = silhouette_score(coords, labels3)
labels6 = KMeans(n_clusters=6, random_state=0, n_init=10).fit_predict(coords)
sil6 = silhouette_score(coords, labels6)
k3_better = sil3 > sil6
print("k=3 실루엣:", round(sil3, 3), " k=6 실루엣:", round(sil6, 3))
print("k=3 이 더 큰가?", bool(k3_better))

In [ ]:
# [자가채점]
assert bool(k3_better) == True
assert k3_better == (sil3 > sil6)   # sil3·sil6 를 실제로 구해 비교했는지
assert 0.3 < sil3 < 0.8 and 0.2 < sil6 < 0.8
print("✅ 문제12 통과!")

### 해설 — 문제 12
- **접근법**: k 를 바꿔 가며 실루엣을 비교하는 것이 '군집 개수 고르기'의 정석입니다. 헤드라인이 20건뿐인데 6개로 나누면 군집당 서너 건씩만 남아 덩어리가 잘게 부서지고, 실루엣이 떨어집니다.
- **핵심**: 실루엣이 높은 k 가 **실제 분야 수(4)와 꼭 같지는 않습니다**. 여기서도 k=3·k=4 의 점수는 거의 붙어 있어 어느 쪽이라 단정하기 어렵습니다. 실루엣은 '기하학적으로 잘 떨어지는' 묶음을 선호할 뿐이므로, **도메인 지식(분야가 4개라는 사실)과 함께** 보고 최종 판단합니다.
- **주의**: `coords` 는 UMAP 이 이웃 관계를 살려 펼친 좌표라 실루엣 값이 원본보다 크게 나옵니다. **값의 크기가 아니라 k 사이의 순위**를 읽으세요.

---
# 벡터 연산 — 수식으로 직접 계산해 보기

여기까지는 `cosine_similarity` 가 알아서 계산해 주었습니다. 마지막 세 문제에서는 **그 함수가 안에서 무슨 계산을 하는지** 직접 해 봅니다. 교안 3-1 절의 식 그대로입니다.

## 13. 내적 · 길이 · 코사인 유사도 손으로 계산하기
**배경**: 코사인 유사도는 **내적 ÷ (길이 × 길이)** 입니다. 768차원은 눈으로 못 따라가니 **2차원 벡터 두 개**로 식을 그대로 옮겨 보고, 라이브러리 값과 같은지 맞춰 봅니다.

**요구사항**:
- `a_vec = [3, 4]`, `b_vec = [4, 3]` 두 리스트를 만드세요.
- 두 벡터의 **내적**(같은 칸끼리 곱해 전부 더한 값)을 변수 `dot_ab` 에 담으세요.
- 각 벡터의 **길이(노름)** — 각 칸을 제곱해 더한 뒤 제곱근 — 을 변수 `norm_a`, `norm_b` 에 담으세요.
- 위 셋으로 **코사인 유사도**를 계산해 변수 `cos_ab` 에 담으세요.
- 같은 계산을 **함수 `my_cos(u, v)`** 로도 만드세요 — 두 리스트를 받아 코사인 유사도를 돌려주며, **칸이 몇 개든**(2개든 3개든) 동작해야 합니다.
- `cosine_similarity` 없이 **파이썬 기본 연산**(`sum`, `**`)만으로 구합니다.

**예시**
```
dot_ab            →  두 벡터의 내적 (숫자 하나)
norm_a            →  a_vec 의 길이 (숫자 하나)
cos_ab            →  dot_ab / (norm_a * norm_b)
my_cos([1, 0], [0, 1])  →  0.0   (직각인 두 벡터)
```
<details><summary>힌트</summary>

```text
접근방법:
- 내적·길이·코사인을 순서대로 구한다. 제곱근은 ** 0.5 로 낼 수 있다.
- 함수는 그 세 줄을 그대로 옮기되, 길이를 len(u) 로 받아 어떤 차원에서도 돌게 만든다.

세부구현:
1. 두 리스트를 만든다
2. 같은 위치끼리 곱한 값을 모두 더해 내적을 구한다(range 와 sum 을 함께 쓰면 짧다)
3. 각 칸을 제곱해 더한 뒤 0.5 제곱해 길이를 구한다
4. 내적을 두 길이의 곱으로 나눈다
5. 같은 계산을 my_cos(u, v) 함수 안에 넣는다
```

</details>

In [ ]:
a_vec = [3, 4]
b_vec = [4, 3]

dot_ab = sum(a_vec[i] * b_vec[i] for i in range(len(a_vec)))
norm_a = sum(x ** 2 for x in a_vec) ** 0.5
norm_b = sum(x ** 2 for x in b_vec) ** 0.5
cos_ab = dot_ab / (norm_a * norm_b)


def my_cos(u, v):
    d = sum(u[i] * v[i] for i in range(len(u)))
    nu = sum(x ** 2 for x in u) ** 0.5
    nv = sum(x ** 2 for x in v) ** 0.5
    return d / (nu * nv)


print("내적:", dot_ab, " 길이:", norm_a, norm_b)
print("코사인 유사도:", cos_ab)
print("my_cos 로 다시:", my_cos(a_vec, b_vec))

In [ ]:
# [자가채점]
assert abs(cos_ab - float(cosine_similarity([a_vec], [b_vec])[0][0])) < 1e-9
assert abs(dot_ab - norm_a * norm_b * cos_ab) < 1e-9   # 세 값의 아귀가 맞는지
assert abs(norm_a - norm_b) < 1e-9                     # 이 두 벡터는 길이가 같다
assert 20 < dot_ab < 30 and 4 < norm_a < 6
assert abs(my_cos(a_vec, b_vec) - cos_ab) < 1e-9
assert abs(my_cos([1, 0], [0, 1]) - 0.0) < 1e-9        # 직각이면 0
assert abs(my_cos([1, 2, 2], [2, 4, 4]) - 1.0) < 1e-9  # 3칸이어도, 방향이 같으면 1
print("✅ 문제13 통과!")

### 해설 — 문제 13
- **접근법**: 내적 $\sum a_i b_i = 3{\times}4 + 4{\times}3 = 24$, 길이 $\sqrt{3^2+4^2} = 5$(두 벡터 모두), 따라서 $\cos\theta = 24 / (5{\times}5) = 0.96$ 입니다. `cosine_similarity` 도 정확히 이 계산을 하므로 값이 소수점 아래까지 같습니다.
- **함수로 만드는 이유**: 식에 차원 수가 들어 있지 않으므로, `len(u)` 만 쓰면 2칸이든 768칸이든 같은 함수가 그대로 동작합니다. 자가채점이 3칸짜리 벡터로도 검사하는 것이 그 확인입니다.
- **흔한 실수**: 1) 같은 칸끼리가 아니라 모든 조합을 곱함 2) 제곱근을 빼먹고 제곱합을 길이로 씀 3) 길이의 **합**으로 나눔(곱이 맞습니다).
- **핵심**: 값이 0.96 으로 1 에 가까운 것은 두 화살표의 각이 약 16도로 좁다는 뜻입니다.

## 14. 768차원 임베딩으로 같은 계산 하기
**배경**: 식에는 차원 수의 제한이 없습니다. 문제 13과 **똑같은 계산**을 768칸짜리 실제 임베딩에 하면 문제 3에서 `cosine_similarity` 로 구한 값이 그대로 나와야 합니다.

**요구사항**:
- 문제 2의 `embeddings` 에서 10번·11번 벡터를 꺼내 씁니다.
- 두 벡터의 내적을 **`@` 연산자**로 구해 변수 `dot_10_11` 에 담으세요(`embeddings[10] @ embeddings[11]`).
- 10번 벡터의 길이를 `np.linalg.norm` 으로 구해 변수 `norm_10` 에, 11번 것을 `norm_11` 에 담으세요.
- 세 값으로 코사인 유사도를 계산해 변수 `cos_manual` 에 담으세요.
- 문제 3의 `sim_same` 과 **같은 값**이 나오는지 확인하세요.

**예시**
```
cos_manual  →  dot_10_11 / (norm_10 * norm_11)   (sim_same 과 같은 값)
```
<details><summary>힌트</summary>

```text
접근방법:
- 문제 13과 같은 식을 numpy 함수로 옮긴다. 길이는 1 이 아니다(이 모델은 정규화하지 않는다).

세부구현:
1. embeddings 에서 10번·11번 벡터를 꺼낸다
2. @ 로 내적을, np.linalg.norm 으로 각 길이를 구한다
3. 내적을 두 길이의 곱으로 나눠 cos_manual 에 담는다
```

</details>

In [ ]:
dot_10_11 = float(embeddings[10] @ embeddings[11])
norm_10 = float(np.linalg.norm(embeddings[10]))
norm_11 = float(np.linalg.norm(embeddings[11]))
cos_manual = dot_10_11 / (norm_10 * norm_11)

print("내적:", round(dot_10_11, 3), " 길이:", round(norm_10, 3), round(norm_11, 3))
print("직접 계산한 코사인:", round(cos_manual, 6))
print("문제 3의 sim_same :", round(sim_same, 6))

In [ ]:
# [자가채점]
assert abs(cos_manual - float(sim_same)) < 1e-5      # 라이브러리 값과 일치해야 한다
assert abs(dot_10_11 - float(embeddings[10] @ embeddings[11])) < 1e-4
assert norm_10 > 1.5 and norm_11 > 1.5               # 이 모델은 길이 1 로 정규화하지 않는다
print("✅ 문제14 통과!")

### 해설 — 문제 14
- **접근법**: 문제 13의 식을 그대로 numpy 로 옮겼을 뿐입니다. `@` 가 768칸을 곱해 더하고, `np.linalg.norm` 이 제곱합의 제곱근을 냅니다. 결과가 `sim_same` 과 소수점 아래까지 같습니다.
- **핵심**: 벡터 길이가 **1 이 아니라는 점**을 눈으로 확인하게 됩니다. 이 모델은 정규화된 벡터를 내놓지 않으므로, 내적만 써서 비교하면 코사인과 순위가 달라질 수 있습니다.
- **흔한 실수**: `embeddings[10:11]`(2차원 배열)을 그대로 `@` 에 넣어 모양 오류가 나는 경우. 여기서는 `embeddings[10]`(1차원 벡터)을 씁니다. 또 `@` 는 **numpy 배열 전용**이라 문제 13의 파이썬 리스트에는 쓸 수 없습니다 — 거기서는 `sum` 으로 직접 더했습니다.

## 15. L2 정규화 — 길이를 1 로 맞추면 내적이 곧 코사인
**배경**: 벡터를 **자기 길이로 나누면** 방향은 그대로인 채 길이가 1 이 됩니다(L2 정규화). 그러면 코사인 식의 분모가 1 이 되어 **내적만 계산해도 코사인 값이 나옵니다** — 벡터 DB 가 임베딩을 미리 정규화해 두는 이유입니다.

**요구사항**:
- 문제 14의 `norm_10`·`norm_11` 을 이용해 10번·11번 벡터를 각각 길이 1 로 만들어 변수 `unit_10`, `unit_11` 에 담으세요(벡터를 자기 길이로 나눕니다).
- 두 단위 벡터의 **내적**을 `@` 로 구해 변수 `dot_unit` 에 담으세요.
- `dot_unit` 이 문제 14의 `cos_manual` 과 같은지 확인하세요.

**예시**
```
np.linalg.norm(unit_10)  →  1.0
dot_unit                 →  cos_manual 과 같은 값
```
<details><summary>힌트</summary>

```text
접근방법:
- 정규화는 나눗셈 한 번이다. numpy 배열은 숫자로 나누면 모든 칸이 한꺼번에 나뉜다.

세부구현:
1. embeddings[10] 을 norm_10 으로 나눠 unit_10 에 담는다(11번도 같은 방식)
2. @ 로 두 단위 벡터의 내적을 구해 dot_unit 에 담는다
3. cos_manual 과 값을 비교해 본다
```

</details>

In [ ]:
unit_10 = embeddings[10] / norm_10
unit_11 = embeddings[11] / norm_11
dot_unit = float(unit_10 @ unit_11)

print("정규화 후 길이:", round(np.linalg.norm(unit_10), 6),
      round(np.linalg.norm(unit_11), 6))
print("정규화 후 내적:", round(dot_unit, 6), " / 코사인:", round(cos_manual, 6))

In [ ]:
# [자가채점]
assert abs(float(np.linalg.norm(unit_10)) - 1.0) < 1e-5
assert abs(float(np.linalg.norm(unit_11)) - 1.0) < 1e-5
assert abs(dot_unit - float(cos_manual)) < 1e-5     # 정규화하면 내적 == 코사인
print("✅ 문제15 통과!")

### 해설 — 문제 15
- **접근법**: $\hat{a} = \vec{a} / \lVert \vec{a} \rVert$ 로 길이를 1 로 맞추면 $\hat{a} \cdot \hat{b} = \dfrac{\vec{a}\cdot\vec{b}}{\lVert\vec{a}\rVert\lVert\vec{b}\rVert} = \cos\theta$ 가 됩니다. 그래서 정규화 후에는 **내적만으로** 코사인을 얻습니다.
- **왜 실무에서 중요한가**: 내적은 나눗셈이 없어 가장 빠릅니다. 벡터를 저장할 때 한 번 정규화해 두면 이후 모든 검색을 내적으로 처리할 수 있어, 대규모 벡터 검색에서 흔히 쓰는 방법입니다.
- **흔한 실수**: 길이가 0 인 벡터(모든 칸이 0)는 나눌 수 없습니다. 또 `norm` 을 구할 때 `axis` 를 잘못 주면 벡터 하나가 아니라 행렬 전체의 길이가 나옵니다.